In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

Question 2. Task Decomposer (Planner + Workers)

Use a PlannerAgent to break down the goal: "Create a business plan for a startup in mental health tech". Then use MarketAgent, TechAgent, and FinancialAgent to address each part.



In [2]:
!pip -q install pyautogen groq

                            USER
                              │
                              ▼
            "Create a business plan for a startup
                  in mental health tech"
                              │
                              ▼
                      PlannerAgent
                (Breaks task into subtasks)
                              │
              ┌───────────────┼─────────────────┐
              ▼               ▼                 ▼
        MarketAgent      TechAgent      FinancialAgent
              │               │                 │
              └───────────────┼─────────────────┘
                              ▼
                    Final Business Plan

In [4]:
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],
        }
    ],
    "temperature": 0.4,
}

Create Agents

In [6]:
!pip install pyautogen==0.2.35

Create Agents

In [18]:
from autogen import ConversableAgent

planner = ConversableAgent(
    name="PlannerAgent",
    system_message="""
You are a project planner.

Break the user's goal into three tasks:
1. Market Research
2. Technology Plan
3. Financial Plan

Return only the task list.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

market = ConversableAgent(
    name="MarketAgent",
    system_message="""
You are a Market Research Expert.

Analyze:
- Target customers
- Competitors
- Market opportunity
- SWOT Analysis

Return a detailed report.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

tech = ConversableAgent(
    name="TechAgent",
    system_message="""
You are a Technology Consultant.

Create the technical architecture including:
- AI Technologies
- Backend
- Frontend
- Database
- Cloud
- Security
- Development Roadmap
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

finance = ConversableAgent(
    name="FinancialAgent",
    system_message="""

You are a Startup Financial Advisor.

Prepare a financial plan in Indian Rupees (₹ INR) only.

Include:
- Startup Cost (₹)
- Revenue Model
- Pricing (₹)
- Monthly Expenses (₹)
- Break-even Analysis
- Funding Strategy

Assume the startup is based in India.
Do not use USD ($). Use ₹ INR only.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)



| **Agent Name**     | **Role**                  | **System Message / Responsibility**                                                                                                                  | **Input**                      | **Output**                                     |
| ------------------ | ------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------ | ---------------------------------------------- |
| **PlannerAgent**   | Project Planner           | Breaks the user's main goal into three subtasks: **Market Research**, **Technology Plan**, and **Financial Plan**. Returns only the task list.       | User's business goal           | List of tasks for worker agents                |
| **MarketAgent**    | Market Research Expert    | Performs market analysis by identifying **target customers, competitors, market opportunities, and SWOT analysis**.                                  | Business goal + planner's task | Detailed market research report                |
| **TechAgent**      | Technology Consultant     | Designs the technical solution, including **AI technologies, backend, frontend, database, cloud infrastructure, security, and development roadmap**. | Business goal + planner's task | Technical architecture and implementation plan |
| **FinancialAgent** | Startup Financial Advisor | Creates the financial strategy, including **startup cost, revenue model, pricing, operating expenses, break-even analysis, and funding strategy**.   | Business goal + planner's task | Financial plan and investment analysis         |


| **Parameter**      | **Value**                                               | **Purpose**                                                            |
| ------------------ | ------------------------------------------------------- | ---------------------------------------------------------------------- |
| `name`             | PlannerAgent / MarketAgent / TechAgent / FinancialAgent | Unique identifier for each agent.                                      |
| `system_message`   | Agent-specific instructions                             | Defines the agent's expertise and responsibilities.                    |
| `llm_config`       | `llm_config`                                            | Connects the agent to the Groq LLM configuration.                      |
| `human_input_mode` | `"NEVER"`                                               | Prevents the agent from requesting manual user input during execution. |


Agent Workflow

| **Step** | **Agent**      | **Task**                                   | **Output**             |
| -------- | -------------- | ------------------------------------------ | ---------------------- |
| 1        | PlannerAgent   | Decomposes the business goal into subtasks | Task list              |
| 2        | MarketAgent    | Performs market research                   | Market Analysis        |
| 3        | TechAgent      | Designs the technical architecture         | Technology Plan        |
| 4        | FinancialAgent | Creates the financial strategy             | Financial Plan         |
| 5        | Final Output   | Combines all reports                       | Complete Business Plan |


        User Goal
            │
            ▼
        PlannerAgent
            │
        ┌────────────────┐
        ▼                ▼
        MarketAgent   TechAgent   FinancialAgent
            │             │              │
            └───────┬─────┴──────────────┘
                    ▼
          Final Business Plan

User Goal

In [8]:
goal = "Create a business plan for a startup in mental health tech."

Planner Breaks Down the Goal

Remove the warning

In [10]:
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],

            # Optional pricing values (example)
            "price": [0.0, 0.0]
        }
    ],
    "temperature": 0.4,
}

Planner Breaks Down the Goal

In [11]:
planner_reply = planner.generate_reply(
    messages=[
        {
            "role": "user",
            "content": goal
        }
    ]
)

print("="*70)
print("PLANNER AGENT")
print("="*70)
print(planner_reply)

PLANNER AGENT
1. Market Research: Identify target audience, assess competitors, and determine market demand for mental health tech solutions.
2. Technology Plan: Develop a platform or application for mental health support, including features such as therapy sessions, mood tracking, and resource libraries.
3. Financial Plan: Establish funding requirements, create a revenue model, and outline projected expenses and growth projections for the startup.


Market Agent

In [ ]:
market_reply = market.generate_reply(
    messages=[
        {
            "role": "user",
            "content": f"""
Business Goal:

{goal}

Planner Tasks:

{planner_reply}

Prepare only the Market Analysis.
"""
        }
    ]
)

print("="*70)
print("MARKET AGENT")
print("="*70)
print(market_reply)

| **Code**                              | **Purpose**          | **Description**                                                                                      |
| ------------------------------------- | -------------------- | ---------------------------------------------------------------------------------------------------- |
| `market.generate_reply()`             | Generate AI response | Calls the **MarketAgent** to generate a market analysis using the Groq LLM.                          |
| `messages=[...]`                      | Input conversation   | Passes the input message(s) to the agent in chat format.                                             |
| `"role": "user"`                      | Message sender       | Indicates that the message is from the user.                                                         |
| `"content": f"""..."""`               | Prompt               | Creates a formatted prompt containing the business goal, planner tasks, and specific instructions.   |
| `{goal}`                              | Business Goal        | Inserts the original user goal (e.g., *Create a business plan for a startup in mental health tech*). |
| `{planner_reply}`                     | Planner Output       | Inserts the task list generated by the **PlannerAgent**.                                             |
| `"Prepare only the Market Analysis."` | Task instruction     | Tells the MarketAgent to focus **only on market analysis** and ignore other tasks.                   |
| `market_reply`                        | Store output         | Saves the MarketAgent's response for later use.                                                      |
| `print("="*70)`                       | Separator            | Prints a horizontal line for better output readability.                                              |
| `print("MARKET AGENT")`               | Heading              | Displays the title of the current agent's output.                                                    |
| `print(market_reply)`                 | Display result       | Prints the market analysis generated by the MarketAgent.                                             |


Tech Agent

In [13]:
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],

            # Optional pricing values (example)
            "price": [0.0, 0.0]
        }
    ],
    "temperature": 0.4,
}

In [14]:
tech_reply = tech.generate_reply(
    messages=[
        {
            "role": "user",
            "content": f"""
Business Goal:

{goal}

Planner Tasks:

{planner_reply}

Prepare only the Technology Plan.
"""
        }
    ]
)

print("="*70)
print("TECH AGENT")
print("="*70)
print(tech_reply)

TECH AGENT
**Technology Plan for Mental Health Tech Startup**

### Overview

Our mental health tech startup aims to create a comprehensive platform that provides accessible and personalized support for individuals seeking mental wellness. The platform will offer a range of features, including virtual therapy sessions, mood tracking, and resource libraries, to cater to the diverse needs of our target audience.

### Technical Architecture

#### AI Technologies

* **Natural Language Processing (NLP)**: Utilize NLP to analyze user input and provide personalized recommendations for therapy sessions, mood tracking, and resource libraries.
* **Machine Learning (ML)**: Implement ML algorithms to identify patterns in user behavior and provide predictive insights for mental health support.
* **Chatbots**: Develop chatbots to offer 24/7 support and guidance for users, using NLP and ML to provide personalized responses.

#### Backend

* **Programming Language**: Use Node.js as the primary programm

Financial Agent

In [16]:
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],

            # Optional pricing values (example)
            "price": [0.0, 0.0]
        }
    ],
    "temperature": 0.4,
}

In [19]:
finance_reply = finance.generate_reply(
    messages=[
        {
            "role": "user",
            "content": f"""
Business Goal:

{goal}

Planner Tasks:

{planner_reply}

Prepare only the Financial Plan.
"""
        }
    ]
)

print("="*70)
print("FINANCIAL AGENT")
print("="*70)
print(finance_reply)

FINANCIAL AGENT
**Financial Plan for Mental Health Tech Startup**

**Startup Cost:**
The estimated startup cost for the mental health tech platform is ₹50,00,000, broken down into:

* Technology development: ₹20,00,000
* Marketing and advertising: ₹10,00,000
* Hiring and training staff: ₹8,00,000
* Office setup and infrastructure: ₹5,00,000
* Miscellaneous (licenses, etc.): ₹7,00,000

**Revenue Model:**
The revenue model for the mental health tech platform will be based on the following streams:

* Subscription-based model: Offer users a monthly or yearly subscription to access premium features, such as therapy sessions and personalized coaching.
* Advertising: Partner with relevant businesses to display non-intrusive ads within the platform.
* Partnerships: Collaborate with healthcare providers, insurance companies, and wellness centers to offer bundled services.

**Pricing:**
The pricing for the mental health tech platform will be as follows:

* Basic plan (mood tracking, resource li

Final Business Plan

In [21]:
market_reply = market.generate_reply(
    messages=[
        {
            "role": "user",
            "content": f"""
Business Goal:

{goal}

Planner Tasks:

{planner_reply}

Prepare only the Market Analysis.
"""
        }
    ]
)

print("="*70)
print("MARKET AGENT")
print("="*70)
print(market_reply)

MARKET AGENT
**Market Analysis for Mental Health Tech Startup**

### Target Customers

The target audience for our mental health tech startup includes:

1. **Demographics**: Individuals aged 18-45, with a focus on millennials and Gen Z, who are more likely to adopt digital solutions for mental health support.
2. **Psychographics**: People experiencing stress, anxiety, depression, or other mental health concerns, as well as those seeking preventive care and wellness.
3. **Pain Points**: Individuals facing barriers to traditional mental health services, such as lack of access, high costs, or social stigma.
4. **Preferred Communication Channels**: Digital natives who are comfortable with online platforms, mobile apps, and social media.

### Competitors

Key competitors in the mental health tech space include:

1. **Teletherapy Platforms**: Companies like BetterHelp, Talkspace, and 7 Cups, which offer online counseling and therapy sessions.
2. **Mental Health Apps**: Apps like Calm, Headsp

In [22]:
print("\n")
print("="*80)
print("FINAL BUSINESS PLAN")
print("="*80)

print("\nBusiness Goal")
print(goal)

print("\n")
print("="*80)
print("MARKET ANALYSIS")
print("="*80)
print(market_reply)

print("\n")
print("="*80)
print("TECHNOLOGY PLAN")
print("="*80)
print(tech_reply)

print("\n")
print("="*80)
print("FINANCIAL PLAN")
print("="*80)
print(finance_reply)



FINAL BUSINESS PLAN

Business Goal
Create a business plan for a startup in mental health tech.


MARKET ANALYSIS
**Market Analysis for Mental Health Tech Startup**

### Target Customers

The target audience for our mental health tech startup includes:

1. **Demographics**: Individuals aged 18-45, with a focus on millennials and Gen Z, who are more likely to adopt digital solutions for mental health support.
2. **Psychographics**: People experiencing stress, anxiety, depression, or other mental health concerns, as well as those seeking preventive care and wellness.
3. **Pain Points**: Individuals facing barriers to traditional mental health services, such as lack of access, high costs, or social stigma.
4. **Preferred Communication Channels**: Digital natives who are comfortable with online platforms, mobile apps, and social media.

### Competitors

Key competitors in the mental health tech space include:

1. **Teletherapy Platforms**: Companies like BetterHelp, Talkspace, and 7 Cups,